In [ ]:
!pip install transformers datasets trl peft bitsandbytes accelerate -q

In [ ]:
from datasets import load_dataset

dataset = load_dataset("iamtarun/python_code_instructions_18k_alpaca")

def format_sample(x):
    return {"text": x["prompt"]}

dataset = dataset.map(format_sample, remove_columns=dataset["train"].column_names)

full = dataset["train"]
total = len(full)
chunk_size = total // 5
chunks = [full.select(range(i * chunk_size, (i + 1) * chunk_size)) for i in range(5)]
print(f"Total: {total} | Each chunk: {chunk_size} samples")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "Qwen/Qwen2.5-Coder-0.5B-Instruct"
model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.bfloat16, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
from trl import SFTConfig, SFTTrainer
from huggingface_hub import login

login(token="API_here")

for i, chunk in enumerate(chunks):
    print(f"\n--- Training chunk {i+1}/5 ({len(chunk)} samples) ---")

    config = SFTConfig(
        output_dir=f"./codealpaca-qwen-chunk{i+1}",
        num_train_epochs=1,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        bf16=True,
        fp16=False,
        optim="adamw_torch",
        dataloader_num_workers=2,
        logging_steps=50,
        save_steps=200,
        dataset_text_field="text",
        max_length=512,
        completion_only_loss=False,
    )

    trainer = SFTTrainer(
        model=model,
        train_dataset=chunk,
        args=config,
        processing_class=tokenizer,
    )

    trainer.train()

    model.push_to_hub("HF_username_here/codealpaca-qwen-lora")
    tokenizer.push_to_hub("HF_username_here/codealpaca-qwen-lora")
    print(f"Chunk {i+1}/5 saved")

print("\nAll done!")

In [ ]:
model.eval()

inputs = tokenizer(
    "### Instruction:\nWrite a Python function to check if a number is prime\n\n### Response:\n",
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=300,
    temperature=0.3,
    do_sample=True,
    top_p=0.9,
    repetition_penalty=1.3,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))